## Output Parsers - Practice Notebook (VS Code)
This notebook corresponds to the **04-Output Parsers** documentation page. Follow the concepts and run the code cells.

### 🛠️ Setup Environment & Load API Keys

In [ ]:
# Load local environment variables from .env
import os
from dotenv import load_dotenv
load_dotenv()

### 1. Introduction to Output Parsers

Output Parsers take the raw text output from an LLM and structure it into a usable programming data type (like a string, list, dictionary, or validated Pydantic model).

### 2. Using Output Parsers

#### 2.1 StrOutputParser
Converts the output of a chat model (`AIMessage`) into a clean, raw string.

In [ ]:
print("--- Example 1: Standalone Usage ---")
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
# Pass an AIMessage object to extract its content
response = parser.invoke(AIMessage(content="Hello World"))
print("Output:", response)
print("Type:", type(response))

In [ ]:
print("--- Example 2: Complete LCEL Chain with LLM ---")
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize model, prompt and parser
model = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
prompt = PromptTemplate.from_template("Tell me a one-line joke about {topic}.")

# Chain components together
chain = prompt | model | StrOutputParser()

# Invoke the chain
response = chain.invoke({"topic": "robots"})
print("Output:", response)
print("Type:", type(response))

#### 2.2 JsonOutputParser
Parses JSON-formatted strings generated by LLMs into a native Python dictionary.

In [ ]:
print("--- Example 1: Standalone Usage ---")
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
response = parser.invoke("""
{
   "name": "John Doe",
   "age": 30,
   "occupation": "Software Engineer"
}
""")
print("Output:", response)
print("Type:", type(response))

In [ ]:
print("--- Example 2: Complete LCEL Chain with LLM ---")
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

model = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
prompt = PromptTemplate.from_template(
    "Return a JSON object containing the capital and population of {country}. Output ONLY valid JSON."
)

# Chain components
chain = prompt | model | JsonOutputParser()

# Invoke
response = chain.invoke({"country": "Japan"})
print("Output:", response)
print("Type:", type(response))

#### 2.3 PydanticOutputParser
Validates the output against a Pydantic model definition. This ensures type safety and field presence.

In [ ]:
print("--- Example 1: Standalone Usage ---")
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser

# Define the schema
class Person(BaseModel):
    name: str
    age: int
    occupation: str

parser = PydanticOutputParser(pydantic_object=Person)
response = parser.invoke("""
{
   "name": "John Doe",
   "age": 30,
   "occupation": "Software Engineer"
}
""")
print("Output:", response)
print("Type:", type(response))
print("Name:", response.name)

In [ ]:
print("--- Example 2: Complete LCEL Chain with LLM ---")
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

class CountryInfo(BaseModel):
    capital: str = Field(description="The capital city of the country")
    official_languages: list[str] = Field(description="List of official languages")

# Setup model and parser
model = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
parser = PydanticOutputParser(pydantic_object=CountryInfo)

# Include formatting instructions in prompt
prompt = PromptTemplate.from_template(
    "Provide information about {country}.\n{format_instructions}"
)

chain = prompt | model | parser

# Invoke
response = chain.invoke({
    "country": "Germany",
    "format_instructions": parser.get_format_instructions()
})
print("Output:", response)
print("Type:", type(response))

#### 2.4 List Output Parser (CommaSeparatedListOutputParser)
Splits comma-separated lists generated by the model into a Python list of strings.

In [ ]:
print("--- Example 1: Standalone Usage ---")
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()
response = parser.invoke("Python, Java, JavaScript, Go, Rust")
print("Output:", response)
print("Type:", type(response))

In [ ]:
print("--- Example 2: Complete LCEL Chain with LLM ---")
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

model = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
parser = CommaSeparatedListOutputParser()

prompt = PromptTemplate.from_template(
    "List the 3 largest cities in {country} as a comma-separated list.\n{format_instructions}"
)

chain = prompt | model | parser

# Invoke
response = chain.invoke({
    "country": "India",
    "format_instructions": parser.get_format_instructions()
})
print("Output:", response)
print("Type:", type(response))

### 2.5 `.invoke()` vs `.parse()`

You can execute parsers using either `.invoke()` or `.parse()`:

* **`.invoke()`** (Recommended)
  - **When to Use**: Standard practice in LCEL flows.
  - **Input Type**: `str` or `AIMessage`.
  - **LCEL Chain Support**: **Yes** (e.g., `chain = prompt | model | parser`).

* **`.parse()`**
  - **When to Use**: Legacy / standalone python scripts.
  - **Input Type**: `str`.
  - **LCEL Chain Support**: **No** (throws an error if piped).

### 3. Practice Exercises

#### Practice 1: Comma Separated List Parsing
Create a prompt template that requests the model to list the top 3 programming languages for web development, and chain it with the `CommaSeparatedListOutputParser` to obtain a Python list.

<details>
<summary><b>Click to view solution</b></summary>

```python
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")
parser = CommaSeparatedListOutputParser()

prompt = PromptTemplate(
    template="List 3 top programming languages for {use_case} as a comma-separated list.",
    input_variables=["use_case"]
)

chain = prompt | llm | parser
result = chain.invoke({"use_case": "web development"})
print(result)
print(type(result))
```
</details>

In [ ]:
# Write your code here